In [1]:
# run once
!pip install -q --upgrade transformers accelerate bitsandbytes sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 46.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.9 MB/s eta 0:00:00:00:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
!nvidia-smi -L


torch: 2.10.0+cu128 cuda available: True
GPU 0: Tesla T4 (UUID: GPU-2a7dfbdd-660c-cf95-0a18-720786b4d082)
GPU 1: Tesla T4 (UUID: GPU-bba61df4-532e-4ac0-0019-343a7f6ae2ad)


In [5]:
import os, random
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm.auto import tqdm
random.seed(42); np.random.seed(42)


In [8]:
# Diagnostic: inspect dataset schemas and example rows
from datasets import load_dataset
import pandas as pd

med_l = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")
med_a = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial")
pubmed = load_dataset("pubmed_qa", "pqa_labeled", split="train")

print("MedHallu labeled features:", med_l['train'].column_names if 'train' in med_l else med_l.column_names)
print("MedHallu artificial features:", med_a['train'].column_names if 'train' in med_a else med_a.column_names)
print("PubMed Features", pubmed.column_names)

# show 5 examples from each
print("\nMedHallu labeled sample rows:")
display(med_l['train'].select(range(min(5, len(med_l['train'])))).to_pandas())
print("\nMedHallu artificial sample rows:")
display(med_a['train'].select(range(min(5, len(med_a['train'])))).to_pandas())
print("\nPubMed sample rows:")
display(pubmed.select(range(min(5, len(pubmed)))).to_pandas())


MedHallu labeled features: ['Question', 'Knowledge', 'Ground Truth', 'Difficulty Level', 'Hallucinated Answer', 'Category of Hallucination']
MedHallu artificial features: ['Question', 'Knowledge', 'Ground Truth', 'Difficulty Level', 'Hallucinated Answer', 'Category of Hallucination']
PubMed Features ['pubid', 'question', 'context', 'long_answer', 'final_decision']

MedHallu labeled sample rows:


,Question,Knowledge,Ground Truth,Difficulty Level,Hallucinated Answer,Category of Hallucination
0,Do mitochondria play a role in remodelling lac...,[Programmed cell death (PCD) is the regulated ...,Results depicted mitochondrial dynamics in viv...,medium,Mitochondria regulate the formation of perfora...,Mechanism and Pathway Misattribution
1,Landolt C and snellen e acuity: differences in...,[Assessment of visual acuity depends on the op...,"Using the charts described, there was only a s...",hard,Patients with strabismus amblyopia showed a si...,Incomplete Information
2,"Syncope during bathing in infants, a pediatric...",[Apparent life-threatening events in infants a...,"""Aquagenic maladies"" could be a pediatric form...",hard,Syncope during bathing in infants is a manifes...,Misinterpretation of #Question#
3,Are the long-term results of the transanal pul...,[The transanal endorectal pull-through (TERPT)...,Our long-term study showed significantly bette...,easy,Both transanal and transabdominal pull-through...,Misinterpretation of #Question#
4,Can tailored interventions increase mammograph...,[Telephone counseling and tailored print commu...,The effects of the intervention were most pron...,hard,Tailored text messages were found to be as eff...,Incomplete Information



MedHallu artificial sample rows:


,Question,Knowledge,Ground Truth,Difficulty Level,Hallucinated Answer,Category of Hallucination
0,Are group 2 innate lymphoid cells ( ILC2s ) in...,[Chronic rhinosinusitis (CRS) is a heterogeneo...,"As ILC2s are elevated in patients with CRSwNP,...",easy,Group 2 innate lymphoid cells (ILC2s) are sign...,Misinterpretation of #Question#
1,Does vagus nerve contribute to the development...,[Phosphatidylethanolamine N-methyltransferase ...,Neuronal signals via the hepatic vagus nerve c...,medium,Capsaicin-induced disruption of hepatic affere...,Misinterpretation of #Question#
2,Does psammaplin A induce Sirtuin 1-dependent a...,[Psammaplin A (PsA) is a natural product isola...,PsA significantly inhibited MCF-7/adr cells pr...,medium,Psammaplin A induces Sirtuin 1-independent aut...,Misinterpretation of #Question#
3,Is methylation of the FGFR2 gene associated wi...,[This study examined links between DNA methyla...,We identified a novel biologically plausible c...,hard,Methylation of the FGFR2 gene is inversely cor...,Misinterpretation of #Question#
4,Do tumor-infiltrating immune cell profiles and...,[Tumor microenvironment immunity is associated...,Breast cancer immune cell subpopulation profil...,hard,The levels of CD20-positive B cells in the tum...,Incomplete Information



PubMed sample rows:


,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,17208539,Are the long-term results of the transanal pul...,{'contexts': ['The transanal endorectal pull-t...,Our long-term study showed significantly bette...,no
4,10808977,Can tailored interventions increase mammograph...,{'contexts': ['Telephone counseling and tailor...,The effects of the intervention were most pron...,yes


In [9]:
from datasets import load_dataset

pubmed = load_dataset("pubmed_qa", "pqa_artificial", split="train")

def join_contexts(example):
    ctx = example.get('context')
    if ctx is None:
        text = ""
    elif isinstance(ctx, (list, tuple)):
        text = " ".join(str(x).strip() for x in ctx if x)
    else:
        text = str(ctx).strip()
    return  {"context" :text}

pubmed = pubmed.map(join_contexts, batched=False, remove_columns=['context'])
# map returned dict will replace removed columns with new ones
print(pubmed[0]['context'])


pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

Map:   0%|          | 0/211269 [00:00<?, ? examples/s]

{'contexts': ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions through flow cytometry. ILC2 frequencies, measured as a percentage of CD45(+) cells, were compared across CRS phenotype, endotype

In [10]:
import pandas as pd, random, json
from datasets import load_dataset

random.seed(42)

# Reload datasets (if not in memory)
med_l = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']
med_a = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial")['train']
#pubmed = load_dataset("pubmed_qa", "pqa_artificial", split="train")

# Convert to pandas
df_med_l = med_l.to_pandas()
df_med_a = med_a.to_pandas()
df_pub = pubmed.to_pandas()

# Combine MedHallu splits and mark source
df_med_l['source_split']='labeled'
df_med_a['source_split']='artificial'
df_med = pd.concat([df_med_l, df_med_a], ignore_index=True).reset_index(drop=True)

# Map actual columns:
# MedHallu: 'Question' (question), 'Ground Truth' (answer), 'Knowledge' (context/source if useful)

df_med_std = pd.DataFrame({
    'question': df_med['Question'].astype(str),
    'answer': df_med['Ground Truth'].astype(str),
    'context': df_med['Knowledge'].astype(str),
    'domain': 'medical',
    'source_split': df_med['source_split']
})

df_pubmed_std = pd.DataFrame({
    'question': df_pub['question'].astype(str),
    'answer': df_pub['long_answer'].astype(str),
    'context': df_pub['context'].astype(str),
    'domain':'medical',
    'orig_id': df_pub['pubid']
})

# Basic cleaning: drop empty questions
def clean_df(df, domain_prefix):
    df = df.dropna(subset=['question']).reset_index(drop=True)
    df['question'] = df['question'].astype(str).str.strip()
    df['answer'] = df['answer'].fillna('').astype(str).str.strip()
    df['context'] = df['context'].fillna('').astype(str)
    df = df[df['question']!=''].reset_index(drop=True)
    df['id'] = domain_prefix + "_" + df.index.astype(str)
    return df

df_med_std = clean_df(df_med_std, 'med')
df_pubmed_std = clean_df(df_pubmed_std, 'pubmed')

print("After clean — med:", len(df_med_std), "pubmed:", len(df_pubmed_std))
display(df_med_std.head())
display(df_pubmed_std.head())

# Splits per domain (70/15/15)
def split_df(df, seed=42):
    n = len(df)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_train = int(0.7 * n)
    n_dev = int(0.15 * n)
    train_idx = idx[:n_train]
    dev_idx = idx[n_train:n_train+n_dev]
    test_idx = idx[n_train+n_dev:]
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[dev_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

med_train, med_dev, med_test = split_df(df_med_std)
pubmed_train,pubmed_dev, pubmed_test = split_df(df_pubmed_std)

print("Med splits sizes:", len(med_train), len(med_dev), len(med_test))
print("pubmed split sizes:", len(pubmed_train), len(pubmed_dev), len(pubmed_test))
# Build retrieval corpus: prefer explicit context; fallback to Q+A
corpus_docs = []
def add_doc(doc_id, title, text):
    corpus_docs.append({'id': doc_id, 'title': title, 'text': text})

for df in [med_train, med_dev, med_test]:
    for _, r in df.iterrows():
        text = r['context'] if r['context'].strip()!='' else (r['question'] + "\n\nAnswer: " + r['answer'])
        add_doc(r['id'], 'MedHallu', text)

for df in [pubmed_train, pubmed_dev, pubmed_test]:
    for _, r in df.iterrows():
        text = r['context'] if r['context'].strip()!='' else (r['question'] + "\n\nAnswer: " + r['answer'])
        add_doc(r['id'], 'PubMed', text)


print("Corpus docs:", len(corpus_docs))

# Chunk corpus into passages (300 tokens, overlap 50)
def chunk_text(text, chunk_size=300, overlap=50):
    tokens = text.split()
    chunks=[]
    i=0
    while i < len(tokens):
        chunk = " ".join(tokens[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

corpus_chunks = []
for doc in corpus_docs:
    chunks = chunk_text(str(doc['text']), chunk_size=300, overlap=50)
    for idx, c in enumerate(chunks):
        corpus_chunks.append({'doc_id': doc['id'], 'chunk_id': f"{doc['id']}_{idx}", 'title': doc['title'], 'text': c})

print("Corpus chunks:", len(corpus_chunks))

# Save artifacts
import os
os.makedirs('prepared_data', exist_ok=True)
med_train.to_parquet('prepared_data/med_train.parquet', index=False)
med_dev.to_parquet('prepared_data/med_dev.parquet', index=False)
med_test.to_parquet('prepared_data/med_test.parquet', index=False)
pubmed_train.to_parquet('prepared_data/pubmed_train.parquet', index=False)
pubmed_dev.to_parquet('prepared_data/pubmed_dev.parquet', index=False)
pubmed_test.to_parquet('prepared_data/pubmed_test.parquet', index=False)

with open('prepared_data/corpus_chunks.jsonl','w',encoding='utf-8') as f:
    for c in corpus_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved prepared_data/ and corpus_chunks.jsonl")


After clean — med: 10000 pubmed: 211269


,question,answer,context,domain,source_split,id
0,Do mitochondria play a role in remodelling lac...,Results depicted mitochondrial dynamics in viv...,['Programmed cell death (PCD) is the regulated...,medical,labeled,med_0
1,Landolt C and snellen e acuity: differences in...,"Using the charts described, there was only a s...",['Assessment of visual acuity depends on the o...,medical,labeled,med_1
2,"Syncope during bathing in infants, a pediatric...","""Aquagenic maladies"" could be a pediatric form...",['Apparent life-threatening events in infants ...,medical,labeled,med_2
3,Are the long-term results of the transanal pul...,Our long-term study showed significantly bette...,['The transanal endorectal pull-through (TERPT...,medical,labeled,med_3
4,Can tailored interventions increase mammograph...,The effects of the intervention were most pron...,['Telephone counseling and tailored print comm...,medical,labeled,med_4


,question,answer,context,domain,orig_id,id
0,Are group 2 innate lymphoid cells ( ILC2s ) in...,"As ILC2s are elevated in patients with CRSwNP,...",{'contexts': ['Chronic rhinosinusitis (CRS) is...,medical,25429730,pubmed_0
1,Does vagus nerve contribute to the development...,Neuronal signals via the hepatic vagus nerve c...,{'contexts': ['Phosphatidylethanolamine N-meth...,medical,25433161,pubmed_1
2,Does psammaplin A induce Sirtuin 1-dependent a...,PsA significantly inhibited MCF-7/adr cells pr...,{'contexts': ['Psammaplin A (PsA) is a natural...,medical,25445714,pubmed_2
3,Is methylation of the FGFR2 gene associated wi...,We identified a novel biologically plausible c...,{'contexts': ['This study examined links betwe...,medical,25431941,pubmed_3
4,Do tumor-infiltrating immune cell profiles and...,Breast cancer immune cell subpopulation profil...,{'contexts': ['Tumor microenvironment immunity...,medical,25432519,pubmed_4


Med splits sizes: 7000 1500 1500
pubmed split sizes: 147888 31690 31691
Corpus docs: 221269
Corpus chunks: 292965
Saved prepared_data/ and corpus_chunks.jsonl


In [6]:
import json, os, math, time
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import faiss
DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [8]:
# load the prepared corpus chunks written earlier
corpus = []
with open('/kaggle/input/datasets/adityamodi20/thesis/corpus_chunks.jsonl','r',encoding='utf-8') as f:
    for line in f:
        corpus.append(json.loads(line))
print("Loaded corpus chunks:", len(corpus))
texts = [c['text'] for c in corpus]


Loaded corpus chunks: 292965


In [13]:
# Current model is slow. Try these faster alternatives:

# Option A: Lighter medical model (fastest, good quality)
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"  # Back to MiniLM (baseline fast)

# Option B: Faster medical model (medium speed, medical-tuned)
embed_model_name = "sentence-transformers/allenai-specter"  # Lighter sci papers

# Option C: Distilled medical (fast + domain-aware)
embed_model_name = "sentence-transformers/biomedical-DistilBERT-TAS-B"

# Test with Option B first
embed_model_name = "dmis-lab/biobert-v1.1"
embed_model = SentenceTransformer(embed_model_name, device=DEVICE)

batch_size = 512  # Can go higher with smaller models
emb_list = []
start = time.time()

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    emb = embed_model.encode(
        batch, 
        convert_to_numpy=True, 
        show_progress_bar=False,
        normalize_embeddings=True
    )
    emb_list.append(emb)

embeddings = np.vstack(emb_list).astype('float32')
print("Emb shape:", embeddings.shape, "time(s):", time.time()-start)


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

  0%|          | 0/573 [00:00<?, ?it/s]

Emb shape: (292965, 768) time(s): 7654.6401534080505


In [17]:
faiss.normalize_L2(embeddings)
d = embeddings.shape[1]
nlist = 100                # # inverted lists
m = 16                    # # subquantizers (must divide d)
nbits = 8                 # bits per codebook
quantizer = faiss.IndexFlatIP(d)                 # coarse quantizer
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)   # must train on representative vectors
index.add(embeddings)
print("Index size:", index.ntotal)
# Save index and metadata
faiss.write_index(index, "prepared_data/faiss_index_flat_ip.index")
# Save metadata (corpus mapping)
with open("prepared_data/corpus_meta.jsonl","w",encoding="utf-8") as f:
    for c in corpus:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("Saved index and metadata")


Index size: 292965
Saved index and metadata


In [30]:
# L2 normalize for cosine-sim using inner-product search
faiss.normalize_L2(embeddings)

d = embeddings.shape[1]
# For moderate corpus sizes, IndexFlatIP is fine. For large corpora consider IVF+PQ.
index = faiss.IndexFlatIP(d)
index.add(embeddings)
print("Index size:", index.ntotal)
# Save index and metadata
faiss.write_index(index, "prepared_data/faiss_index_flat_ip.index")
# Save metadata (corpus mapping)
with open("prepared_data/corpus_meta.jsonl","w",encoding="utf-8") as f:
    for c in corpus:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("Saved index and metadata")


Index size: 292965
Saved index and metadata


In [31]:
from pprint import pprint
# simple retrieval function using index and the same embed_model
import re

def preprocess_medical_query(query):
    # Normalize spacing
    query = ' '.join(query.split())
    # Keep domain terms intact (don't lowercase aggressively)
    query = query.strip()
    return query


def retrieve_top_k(query, k=100):
    query = preprocess_medical_query(query)
    q_emb = embed_model.encode(query, convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb.reshape(1,-1))
    scores, ids = index.search(q_emb.reshape(1,-1), k)
    ids = ids[0].tolist()
    scores = scores[0].tolist()
    results = []
    for sc, idx in zip(scores, ids):
        meta = corpus[idx]
        results.append({'score': float(sc), 'doc_id': meta['doc_id'], 'chunk_id': meta['chunk_id'], 'title': meta['title'], 'text': meta['text']})
    return results

# quick test
q = "What are common side effects of aspirin?"
pprint(retrieve_top_k(q, k=50))


[{'chunk_id': 'med_86_0',
  'doc_id': 'med_86',
  'score': 0.8840212821960449,
  'text': "['Sulfasalazine is a widely used anti-inflammatory agent in the "
          'treatment of inflammatory bowel disease and several rheumatological '
          'disorders. Although as many as 20% of treated patients may '
          'experience reversible, dose-dependent side effects, less frequent '
          'but potentially severe, systemic reactions have also been '
          "reported.' 'A severe systemic reaction to sulfasalazine developed "
          'in a 21-year old female with rheumatoid arthritis characterized by '
          'eosinophilia, granulomatous enteritis and myelotoxicity, '
          'cholestatic hepatitis, and seizures. The clinical course and '
          'management of this patient are presented as well as a review of the '
          'incidence and outcome of severe systemic reactions to '
          "sulfasalazine.']",
  'title': 'MedHallu'},
 {'chunk_id': 'med_775_1',
  'doc_id